#### Python Code Optimization and Evaluator using Gradio

##### Features:
- Used Ollama Open Source Models
- Used OpenRouter Open Source Free Models
- Selection from multiple models for Code OPtimization and Evaluation
- Option to Run Original and Generated Optimized Python Code
- Evaluation of Optimized Python Code and its Output and Comparing it with the Original Python Code and its Output
- Streaming Evaluation Result
- Markdown for Evaluation Result
- CSS for Gradio Modern UI

In [ ]:
# Imports
import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from styles import CSS

In [ ]:
# Client Configuration
load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)

In [ ]:
# Models and Default Python Code Setup
models = ["openai/gpt-oss-120b:free", "openai/gpt-oss-20b:free", "qwen2.5-coder:7b", "codellama:7b", "llama3:8b", "qwen/qwen3-coder:free", "meta-llama/llama-3.3-70b-instruct:free", "google/gemma-4-31b-it:free"]
clients = {"openai/gpt-oss-120b:free": openrouter, "openai/gpt-oss-20b:free": openrouter, "qwen2.5-coder:7b": ollama, "codellama:7b": ollama, "llama3:8b": ollama, "qwen/qwen3-coder:free": openrouter, "meta-llama/llama-3.3-70b-instruct:free": openrouter, "google/gemma-4-31b-it:free": openrouter}

default_python = """
numbers = [5, 12, 7, 3, 18, 9, 4, 15, 8, 11]

total = 0

for i in range(len(numbers)):
    total = total + numbers[i]

average = total / len(numbers)

even_numbers = []

for i in range(len(numbers)):
    if numbers[i] % 2 == 0:
        even_numbers.append(numbers[i])

squared_numbers = []

for i in range(len(numbers)):
    squared_numbers.append(numbers[i] * numbers[i])

largest = numbers[0]

for i in range(len(numbers)):
    if numbers[i] > largest:
        largest = numbers[i]

smallest = numbers[0]

for i in range(len(numbers)):
    if numbers[i] < smallest:
        smallest = numbers[i]

print("Total:", total)
print("Average:", average)
print("Even Numbers:", even_numbers)
print("Squared Numbers:", squared_numbers)
print("Largest:", largest)
print("Smallest:", smallest)
"""

# Run Python Code Function
def run_python_code(code):
    globals_dict = {"__builtins__": __builtins__}
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer
    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error Running Python Code: {e}"
    finally:
        sys.stdout = old_stdout
    return output

In [ ]:
# Main System and User Prompts Setup
system_prompt = """
You are an expert Python performance engineer.

Your task is to optimize Python code while preserving identical behavior and output.

Requirements:
- Return ONLY valid Python code.
- Do NOT include explanations, markdown, or code fences.
- Preserve the exact functionality and output.
- Improve performance where possible.
- Reduce unnecessary complexity and code length.
- Remove redundant variables, loops, imports, and computations.
- Apply Pythonic best practices.
- Fix syntax, formatting, and style issues.
- Keep the code readable and maintainable.
- Do not change external behavior.
- If no optimization is possible, return a cleaned and properly formatted version.
"""

def user_prompt(python):
    return f"""
        Optimize the following Python code.

        Goals (highest priority first):
        1. Preserve identical behavior and output.
        2. Improve execution speed.
        3. Reduce memory usage.
        4. Simplify and shorten the code.
        5. Improve formatting and readability.
        6. Add Python Comments and Docstrings if needed.

        Return ONLY the final optimized Python code.

        Python Code:
        ```python
        {python}
        ```
    """

def messages(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt(python)}
    ]

def optimization(model, python):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages(python), reasoning_effort=reasoning_effort)
    optimized_code = response.choices[0].message.content
    optimized_code = optimized_code.replace('```python','').replace('```','')
    return optimized_code

In [ ]:
# Evaluation System and User Prompts Setup
evaluation_system_prompt = """
You are a senior Python software engineer and code reviewer.

Your task is to compare an original Python program and an optimized Python program.

Evaluate the optimized version across the following dimensions:

1. Correctness
   - Does it preserve the original behavior?
   - Does it preserve the original output?
   - Are there any logic changes?

2. Output Equivalence
   - Compare the original output and optimized output.
   - Determine whether the outputs are identical.
   - Explain any differences.

3. Performance
   - Estimate whether the optimized version is likely faster.
   - Identify removed inefficiencies.
   - Mention any remaining bottlenecks.

4. Code Quality
   - Readability
   - Maintainability
   - Simplicity
   - Pythonic practices

5. Risk Analysis
   - Potential bugs introduced.
   - Edge cases.
   - Hidden behavior changes.
   - If there are no Risks then say "No significant risks identified Successfully."

6. Suggestions
   - Suggest further improvements if applicable.
   - If there are issues, suggest how to fix them.

Provide ratings from 1-10 for:
- Correctness
- Output Equivalence
- Performance
- Readability
- Maintainability
- Overall Quality
- Overall Score(out of 60)

Scoring rules:
- 10 = excellent
- 7-9 = good
- 4-6 = acceptable
- 1-3 = poor

If the optimized code produces identical output and preserves behavior, explicitly state this and say "Outputs Are Successfully Identical".

Respond in concise professional language.
Use Markdown formatting and bullet points where appropriate to structure your response.
"""

def evaluate_results(python_code, optimized_code):
    python_output = run_python_code(python_code)
    optimized_output = run_python_code(optimized_code)

    evaluation_user_prompt = f"""
        Compare the following Python programs.

        ORIGINAL PYTHON CODE:
        ```python
        {python_code}
        ```

        OPTIMIZED PYTHON CODE:
        ```python
        {optimized_code}
        ```

        ORIGINAL OUTPUT:
        {python_output}

        OPTIMIZED OUTPUT:
        {optimized_output}

        Perform the following:

        Determine whether the outputs are identical.
        Determine whether behavior appears preserved.
        Identify performance improvements.
        Identify readability improvements.
        Identify maintainability improvements.
        Identify any mistakes, regressions, risks, or concerns.
        Identify any suggestions for further improvement.
        Give ratings from 1-10 for:
        Correctness
        Output Equivalence
        Performance
        Readability
        Maintainability
        Overall Quality
        Overall Score(out of 60)
        Use Markdown formatting and bullet points where appropriate to structure your response.

        Use the following format:

        OUTPUT COMPARISON:
        ...

        BEHAVIOR ANALYSIS:
        ...

        PERFORMANCE ANALYSIS:
        ...

        CODE QUALITY ANALYSIS:
        ...

        RISKS / ISSUES:
        ...

        SUGGESTIONS:
        ...

        RATINGS:
        - Correctness: X/10
        - Output Equivalence: X/10
        - Performance: X/10
        - Readability: X/10
        - Maintainability: X/10
        - Overall Quality: X/10
        - Overall Score: X/60
        
        FINAL VERDICT:
        ...
    """

    return evaluation_user_prompt
    
def evaluation_messages(python, optimized_code):
    return [
        {"role": "system", "content": evaluation_system_prompt},
        {"role": "user", "content": evaluate_results(python, optimized_code)}
    ]

def evaluation(model, python, optimized_code):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=evaluation_messages(python, optimized_code), stream=True, reasoning_effort=reasoning_effort)
    evaluated_response = ""
    for chunk in response:
        evaluated_response += chunk.choices[0].delta.content or ''
        yield evaluated_response

In [ ]:
# Gradio UI and Setup
with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Python Code Optimization") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (Original)",
                value=default_python,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            optimized_python = gr.Code(
                label=f"Python (Optimized)",
                value="",
                language="python",
                lines=26
            )

    with gr.Row(elem_classes=["controls"], equal_height=True):
        python_run = gr.Button("Run Python", elem_classes=["run-btn"])
        model = gr.Dropdown(models, value=models[0], show_label=False, filterable=False, elem_classes=["dropdowns"])
        optimize = gr.Button(f"Optimize Python Code", elem_classes=["optimize-btn"])
        run_python_output = gr.Button(f"Run Optimized Python", elem_classes=["run-btn"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_output = gr.TextArea(label="Python Result (Original)", lines=8, elem_classes=["python-output"])
        with gr.Column(scale=6):
            optimized_python_output = gr.TextArea(label=f"Python Result (Optimized)", lines=8, elem_classes=["optimized-python-output"])

    with gr.Row(equal_height=True):
            with gr.Column(scale=6):
                evaluate_model = gr.Dropdown(models, value=models[0], show_label=False, filterable=False, elem_classes=["dropdowns"])
            with gr.Column(scale=6):
                evaluate_btn = gr.Button("Evaluate Optimized Python Code", elem_classes=["optimize-btn"])
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            final_evaluation = gr.Markdown(elem_classes=["evaluation-box"])

    optimize.click(fn=optimization, inputs=[model, python], outputs=[optimized_python])
    python_run.click(fn=run_python_code, inputs=[python], outputs=[python_output])
    run_python_output.click(fn=run_python_code, inputs=[optimized_python], outputs=[optimized_python_output])
    evaluate_btn.click(fn=evaluation, inputs=[evaluate_model, python, optimized_python], outputs=[final_evaluation])

ui.launch(inbrowser=True)